# Segment And Flag

This code creates the order window segments as well as the flags for abandonment and recovery. It was created independent of the eda and modeling notebooks using a local python development environment. As such, is wasn't subject to the same cleaning/documenting process the other files experienced.

In [0]:
import json
import pandas as pd
 
# Parse 'items' whether it's a JSON array or a JSON-encoded string of that array
def parse_items(cell: str):
    if pd.isna(cell) or cell.strip() in ("", "[]"):
        return []
    s = cell.strip()
    try:
        # Case 1: already a JSON array
        if s.startswith("[") and s.endswith("]"):
            return json.loads(s)
        # Case 2: JSON string containing the array -> decode twice
        return json.loads(json.loads(s))
    except Exception:
        # Fallback: strip outer quotes and unescape, then parse
        if s.startswith('"') and s.endswith('"'):
            s = s[1:-1].replace('\\"', '"')
        return json.loads(s)
 
GoogleAnalytics = pd.read_csv(
    "clean_google_analytics.csv",
    converters={"items": parse_items},
    # Helps if the CSV used backslash escapes for quotes inside the field
    escapechar="\\"
)
 

In [0]:
GoogleAnalytics['event_ts_utc'] = pd.to_datetime(GoogleAnalytics['event_ts_utc'])
GoogleAnalytics = GoogleAnalytics.sort_values(by=['event_ts_utc'], ascending=[False])


GoogleAnalytics.head(n=5)

In [0]:
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 10000)

In [0]:
# Google anaytics
#GoogleAnalytics = pd.read_csv("google_analytics.csv")
# orders
orders = pd.read_csv("orders.csv")
# sales 
sales = pd.read_csv("sales.csv")
# material
material = pd.read_csv("material.csv")
# customer
customer = pd.read_csv("customer.csv")
# custoff_times
cutoff_times = pd.read_csv("cutoff_times.csv")
# operating hours
operating_hours = pd.read_csv("operating_hours.csv")
# visit plan
visit_plan = pd.read_csv("clean_visit_plan.csv", low_memory=False)

In [0]:
visit_plan = visit_plan.sort_values(by=['customer_id', 'snapshot_ts_utc'], ascending=[True, True])
visit_plan.head(n=500)

# 1. Define the valid recurring frequency values
valid_frequencies = [
    7,
    14,
    21,
    28,
    35,
    42,
    56,
    70,
]

# 2. Filter to only rows that contain these valid frequencies
visit_plan_filtered = visit_plan[visit_plan['frequency'].isin(valid_frequencies)].copy()

# 3. Convert SNAPSHOT_DATE to datetime if it's not already
visit_plan_filtered['snapshot_ts_utc'] = pd.to_datetime(visit_plan_filtered['snapshot_ts_utc'])

# 4. Sort by CUSTOMER_ID, then most recent SNAPSHOT_DATE
visit_plan_filtered = visit_plan_filtered.sort_values(['customer_id', 'snapshot_ts_utc'], ascending=[True, False])

# 5. Keep only the *most recent valid row* per CUSTOMER_ID
visit_plan_latest = visit_plan_filtered.drop_duplicates(subset='customer_id', keep='first')

# 6. Final output
visit_plan_latest.reset_index(drop=True, inplace=True)

visit_plan_latest.head(n=5)


In [0]:
# Make sure ANCHOR_DATE is a datetime type
visit_plan_latest['anchor_ts_utc'] = pd.to_datetime(visit_plan_latest['anchor_ts_utc'])

# Add a column with the day of the week (e.g. Monday, Tuesday)
visit_plan_latest['anchor_day_of_week'] = visit_plan_latest['anchor_ts_utc'].dt.day_name()

# Preview
visit_plan_latest.head()


In [0]:

import pandas as pd
import numpy as np

# --- Input: visit_plan_latest with columns including ANCHOR_DAY_OF_WEEK & FREQUENCY ---
# Example:
# visit_plan_latest = pd.DataFrame({...})

# 1) Clean types
visit_plan_latest = visit_plan_latest.copy()
visit_plan_latest['snapshot_ts_utc'] = pd.to_datetime(visit_plan_latest['snapshot_ts_utc'], errors='coerce')
visit_plan_latest['anchor_ts_utc']   = pd.to_datetime(visit_plan_latest['anchor_ts_utc'],   errors='coerce')
visit_plan_latest['anchor_day_of_week'] = visit_plan_latest['anchor_day_of_week'].astype(str).str.strip()
visit_plan_latest['frequency'] = visit_plan_latest['frequency'].astype(str).str.strip()

# 2) Frequency → days mapping
freq_days = {
    '7': 7,
    '14': 14,
    '21': 21,
    '28': 28,
    '35': 35,
    '42': 42,
    '56': 56,
    '70': 70,
}
# keep only valid frequencies
vp = visit_plan_latest[visit_plan_latest['frequency'].isin(freq_days.keys())].copy()

# 3) Range bounds (inclusive start, exclusive end is fine)
# "2 weeks before the week of 05/27/2024" → start from Monday 2024-05-13
range_start = pd.Timestamp('2024-05-27') - pd.Timedelta(days=14)  # 2024-05-13 (Mon)
# "2 weeks after the week of 05/26/2025"  → through Monday 2025-06-09
range_end   = pd.Timestamp('2025-05-26') + pd.Timedelta(days=14)  # 2025-06-09 (Mon)

# 4) Helper: day name → weekday index (Mon=0..Sun=6)
wk_map = {d: i for i, d in enumerate(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])}
def weekday_index(day_name: str) -> int:
    # be forgiving with case/spaces
    dn = day_name.strip().title()
    return wk_map.get(dn, None)

# 5) Build expanded schedule
records = []
cols = vp.columns.tolist()  # to copy all original columns per expanded row

for _, row in vp.iterrows():
    # resolve weekday and period length in days
    wd = weekday_index(row['anchor_day_of_week'])
    if wd is None:
        continue  # skip if unrecognized day
    period_days = freq_days[row['frequency']]

    # find first cutoff_start on the correct weekday ON OR AFTER range_start
    offset = (wd - range_start.weekday()) % 7
    first_start = range_start + pd.Timedelta(days=offset)

    # generate repeating windows until range_end
    start = first_start
    while start < range_end:
        end = start + pd.Timedelta(days=period_days)
        rec = {c: row[c] for c in cols}
        rec['cutoff_start'] = start.normalize()
        rec['cutoff_end']   = end.normalize()
        records.append(rec)
        # next window starts where the previous ended
        start = end

expanded = pd.DataFrame.from_records(records)

# (Optional) Keep within the exact range if you want to clamp windows:
# expanded = expanded[(expanded['cutoff_end'] > range_start) & (expanded['cutoff_start'] < range_end)]

# Order nicely
expanded = expanded.sort_values(['customer_id', 'cutoff_start']).reset_index(drop=True)

# Show result
expanded.head(20)
print(f"Customers expanded: {expanded['customer_id'].nunique()} | Rows: {len(expanded)}")


# STOP

expanded is clean

In [0]:
import pandas as pd

# --- Assumptions ---
# expanded : schedule table [CUSTOMER_ID, cutoff_start, cutoff_end, ...]
# GoogleAnalytics : GA events [CUSTOMER_ID, EVENT_TIMESTAMP, EVENT_NAME, ...]

# 1) Clean types — make ALL timestamps UTC tz-aware so comparisons work
expanded1 = expanded.copy()
expanded1['cutoff_start'] = pd.to_datetime(expanded1['cutoff_start'],format='ISO8601', utc=True)
expanded1['cutoff_end']   = pd.to_datetime(expanded1['cutoff_end'],format='ISO8601', utc=True)

ga = GoogleAnalytics.copy()
ga['event_ts_utc'] = pd.to_datetime(ga['event_ts_utc'],format='ISO8601', utc=True)

# 2) Keep purchases only (and only the columns we need)
ga_purch = ga.loc[ga['event_name'].eq('purchase'), ['customer_id', 'event_ts_utc']].copy()

# 3) Expand: left join on CUSTOMER_ID, then keep only purchases inside each window
exp_keyed = expanded1.reset_index().rename(columns={'index': 'row_id'})
merged = exp_keyed.merge(ga_purch, on='customer_id', how='left')

# inside-window mask (inclusive)
mask = merged['event_ts_utc'].between(merged['cutoff_start'], merged['cutoff_end'], inclusive='both')

# Rows (row_id, purchase_date) for matches — one per purchase in the window
hits = (merged.loc[mask, ['row_id', 'event_ts_utc']]
              .rename(columns={'event_ts_utc': 'purchase_date'}))

# 4) Reattach to schedule. Left-merge duplicates schedule rows when multiple purchases exist.
# If no purchases for a row_id, you'll get exactly one row with purchase_date = NaT
expanded_with_all_purchases = (
    exp_keyed.merge(hits, on='row_id', how='left')
             .drop(columns='row_id')
)

# Optional: nice ordering
expanded_with_all_purchases = expanded_with_all_purchases.sort_values(
    by=['customer_id', 'cutoff_start', 'purchase_date'], kind='mergesort'
).reset_index(drop=True)

# Preview
expanded_with_all_purchases.head(10)


In [0]:
count = expanded_with_all_purchases['purchase_date'].notna().sum()
print(count)

count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
import pandas as pd

# df is your expanded schedule+purchases dataframe (no new rows to be added)
df = expanded_with_all_purchases.copy()

# GA events prepped
ga = GoogleAnalytics.copy()
ga['event_ts_utc'] = pd.to_datetime(ga['event_ts_utc'], utc=True, errors='coerce')

# Only add_to_cart events (keep only what we need)
ga_cart = ga.loc[ga['event_name'].eq('add_to_cart'), ['customer_id', 'event_ts_utc']].copy()

# Ensure schedule columns are tz-aware datetimes (safe if already)
for col in ['cutoff_start', 'cutoff_end', 'purchase_date']:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

# Key rows so we can aggregate and merge the result back without duplicating rows
exp_keyed = df.reset_index().rename(columns={'index': 'row_id'})

# Join all candidate add_to_cart events by CUSTOMER_ID
merged = exp_keyed.merge(ga_cart, on='customer_id', how='left')

# Build masks per-row for the two cases
no_purchase  = merged['purchase_date'].isna()
has_purchase = ~no_purchase

mask_no_purchase = no_purchase & merged['event_ts_utc'].between(
    merged['cutoff_start'], merged['cutoff_end'], inclusive='both'
)

mask_has_purchase = (
    has_purchase
    & (merged['event_ts_utc'] >= merged['cutoff_start'])
    & (merged['event_ts_utc'] <  merged['purchase_date'])  # strictly before purchase
)

# Candidate add_to_cart timestamps that satisfy the appropriate window rules
merged['cand_added_to_cart'] = merged['event_ts_utc'].where(
    mask_no_purchase | mask_has_purchase
)

# For each original row, take the max qualifying add_to_cart timestamp
added_to_cart_by_row = (
    merged.groupby('row_id', as_index=False)['cand_added_to_cart']
          .max()
          .rename(columns={'cand_added_to_cart': 'added_to_cart'})
)

# Attach back to the original df (no new rows added)
df = (
    exp_keyed.merge(added_to_cart_by_row, on='row_id', how='left')
             .drop(columns='row_id')
)

# Optional: ensure dtype is tz-aware datetime
df['added_to_cart'] = pd.to_datetime(df['added_to_cart'], utc=True, errors='coerce')


In [0]:
count = df['purchase_date'].notna().sum()
print(count)

count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
# import pandas as pd

# # df is your expanded schedule+purchases dataframe (shape must stay the same)
# df = df.copy()

# # Prep GA events
# ga = GoogleAnalytics.copy()
# ga['event_ts_utc'] = pd.to_datetime(ga['event_ts_utc'], utc=True, errors='coerce')

# # Keep only update_cart events
# ga_update = ga.loc[ga['event_name'].eq('update_cart'), ['customer_id', 'event_ts_utc']].copy()

# # Ensure tz-aware datetimes (safe if already set)
# for col in ['cutoff_start', 'cutoff_end', 'purchase_date']:
#     df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

# # Key rows so we can aggregate and merge back 1:1 (no new rows)
# exp_keyed = df.reset_index().rename(columns={'index': 'row_id'})

# # Join all candidate update_cart events by CUSTOMER_ID
# merged = exp_keyed.merge(ga_update, on='customer_id', how='left')

# # Build masks per-row for the two cases
# no_purchase  = merged['purchase_date'].isna()
# has_purchase = ~no_purchase

# # Case A: purchase_date is NaT -> max update_cart in [cutoff_start, cutoff_end]
# mask_no_purchase = merged['event_ts_utc'].between(
#     merged['cutoff_start'], merged['cutoff_end'], inclusive='both'
# ) & no_purchase

# # Case B: purchase_date is not NaT -> max update_cart in [cutoff_start, purchase_date)
# mask_has_purchase = (
#     (merged['event_ts_utc'] >= merged['cutoff_start']) &
#     (merged['event_ts_utc'] <  merged['purchase_date']) &
#     has_purchase
# )

# # Candidate timestamps that satisfy the window rules
# merged['cand_updated_cart'] = merged['event_ts_utc'].where(
#     mask_no_purchase | mask_has_purchase
# )

# # For each original row, take the max qualifying update_cart timestamp
# updated_cart_by_row = (
#     merged.groupby('row_id', as_index=False)['cand_updated_cart']
#           .max()
#           .rename(columns={'cand_updated_cart': 'updated_cart'})
# )

# # Attach back to original df (no extra rows)
# df = (
#     exp_keyed.merge(updated_cart_by_row, on='row_id', how='left')
#              .drop(columns='row_id')
# )

# # Ensure dtype is tz-aware datetime
# df['updated_cart'] = pd.to_datetime(df['updated_cart'], utc=True, errors='coerce')


In [0]:
import pandas as pd

dfEff = df.copy()

# Ensure tz-aware datetimes
for col in ['cutoff_start', 'cutoff_end', 'purchase_date']:
    dfEff[col] = pd.to_datetime(dfEff[col], utc=True, errors='coerce')

# Sort so that "previous row" is well-defined within each CUSTOMER_ID
dfEff = dfEff.sort_values(['customer_id', 'cutoff_start'], kind='mergesort')

# Helper: earlier of two datetimes, treating NaT as missing (returns the other if present)
def earlier(a: pd.Series, b: pd.Series) -> pd.Series:
    return a.where((a <= b) | b.isna(), b)

# 1) effective_end = earlier(cutoff_end, purchase_date)
dfEff['effective_end'] = earlier(dfEff['cutoff_end'], dfEff['purchase_date'])

# 2) First-row initial_start per CUSTOMER_ID = earlier(cutoff_start, purchase_date)
initial_start = earlier(dfEff['cutoff_start'], dfEff['purchase_date'])

# 3) Chain rule per CUSTOMER_ID:
#    - For the first row of each CUSTOMER_ID: effective_start = initial_start
#    - For subsequent rows:                  effective_start = previous row's effective_end
prev_end = dfEff.groupby('customer_id')['effective_end'].shift(1)
first_in_group = dfEff['customer_id'].ne(dfEff['customer_id'].shift())

dfEff['effective_start'] = prev_end
dfEff.loc[first_in_group, 'effective_start'] = initial_start[first_in_group]

# (Optional) If you want to restore original order after computation:
# df = df.sort_index()


In [0]:
count = dfEff['purchase_date'].notna().sum()
print(count)

count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
# Ensure these columns are datetime, in case they aren't already
dfAb = dfEff.copy()
dfAb['purchase_date'] = pd.to_datetime(dfAb.get('purchase_date'), errors='coerce')
dfAb['added_to_cart'] = pd.to_datetime(dfAb.get('added_to_cart'), errors='coerce')
#dfAb['updated_cart']  = pd.to_datetime(dfAb.get('updated_cart'), errors='coerce')

# Create abandoned column with the logic:
dfAb['abandoned'] = False  # default

# Rule 1: If purchase_date exists → NOT abandoned
#df.loc[dfAb['purchase_date'].notna(), 'abandoned'] = False

# Rule 2: If NO purchase AND updated_cart is after add_to_cart → abandoned = True
dfAb.loc[
    (dfAb['purchase_date'].isna()) &
    (dfAb['added_to_cart'].notna()),
    'abandoned'
] = True

# Optional: make it boolean type (True/False)
dfAb['abandoned'] = dfAb['abandoned'].astype(bool)


In [0]:
count = dfAb['purchase_date'].notna().sum()
print(count)

count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
dfAb.head(n=500)

In [0]:
true_count = dfAb['abandoned'].sum()
print(true_count)

In [0]:
# Create purchase_segment as "customer_id.increment"
dfSeg = dfAb.copy()
dfSeg['purchase_segment'] = (
    dfSeg.groupby('customer_id')
      .cumcount() + 1
).astype(str)

dfSeg['purchase_segment'] = dfSeg['customer_id'].astype(str) + '.' + dfSeg['purchase_segment']



In [0]:
count = dfSeg['purchase_date'].notna().sum()
print(count)

count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
import numpy as np
dfPurchorNo = dfSeg.copy()
dfPurchorNo['False_by_purchase'] = np.where(dfPurchorNo['purchase_date'].isna(), 'no purchase', 'purchased')


In [0]:
count = dfPurchorNo['purchase_date'].notna().sum()
print(count)

count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
dfPurchorNo.head(n=500)

In [0]:
import numpy as np
import pandas as pd

dfRec = dfPurchorNo.copy()

# Order within each customer by the numeric suffix of purchase_segment if present, else by cutoff_start
if 'purchase_segment' in dfRec.columns:
    # Extract numeric suffix after the dot (e.g., "500245797.25" -> 25)
    seg_num = (
        dfRec['purchase_segment']
        .astype(str)
        .str.split('.', n=1).str[-1]
        .replace({'nan': np.nan})
        .astype(float)
        .astype('Int64')
    )
    dfRec['_seg_order'] = seg_num
else:
    dfRec['_seg_order'] = pd.Series([pd.NA] * len(dfRec), index=dfRec.index)

# Fallback: where _seg_order is NA, use cutoff_start ordering surrogate
if 'cutoff_start' in dfRec.columns:
    dfRec['_time_order'] = pd.to_datetime(dfRec['cutoff_start'], utc=True)
else:
    # last resort: snapshot or anchor if available
    fallback_cols = [c for c in ['anchor_date', 'effective_start'] if c in dfRec.columns]
    dfRec['_time_order'] = pd.to_datetime(dfRec[fallback_cols[0]], utc=True,) if fallback_cols else pd.Series(pd.NaT, index=dfRec.index)

# Sort by customer then by segment order (or time if missing)
dfRec = dfRec.sort_values(['customer_id', '_seg_order', '_time_order'], kind='mergesort')

# Initialize all as "not applicable"
dfRec['recovered'] = 'not applicable'

def label_group(g: pd.DataFrame) -> pd.Series:
    labels = np.array(['not applicable'] * len(g), dtype=object)
    abandoned_idx = np.where(g['abandoned'].values == True)[0]
    purchased_mask = g['purchase_date'].notna().values

    for i in abandoned_idx:
        # find first purchase AFTER this abandoned row
        if i + 1 >= len(g):
            continue
        after = purchased_mask[i+1:]
        if after.any():
            j = i + 1 + np.argmax(after)  # index of first purchase after i
            # set "intermediate" for rows strictly between i and j (if any)
            if j - i > 1:
                # only set where not already labeled (preserve prior recovered)
                mid_slice = slice(i+1, j)
                mid_idx = np.arange(i+1, j)
                to_set = (np.array(labels[mid_idx]) == 'not applicable')
                labels[mid_idx[to_set]] = 'intermediate'
            # mark the recovery row itself (don’t overwrite if already recovered)
            if labels[j] == 'not applicable':
                labels[j] = 'recovered'
    return pd.Series(labels, index=g.index)

# Apply per customer_id
dfRec['recovered'] = dfRec.groupby('customer_id', sort=False, group_keys=False).apply(label_group)

# Clean up helper columns
dfRec = dfRec.drop(columns=['_seg_order', '_time_order'])

# df now has the new 'recovered' column


In [0]:
count = dfRec['purchase_date'].notna().sum()
print(count)
count1 = ((dfRec['False_by_purchase'] == 'purchased') & (dfRec['purchase_date'].isna())).sum()
print(count1)
count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
dfRec

In [0]:
dfRec.head(n=1000)

In [0]:
import pandas as pd

# Add 5 seconds to effective_start
dfRec['effective_start'] = dfRec['effective_start'] + pd.to_timedelta(5, unit='s')


In [0]:
import pandas as pd
import numpy as np

# # --- 0) Copies and dtypes ---
GA = GoogleAnalytics.copy()
GA['event_ts_utc'] = pd.to_datetime(GA['event_ts_utc'], utc=True)

# Columns we need from df (schedule table). Assumes effective_end already respects purchase_date.
sched = dfRec[['customer_id',
            'effective_start','effective_end',
            'abandoned','purchase_segment','False_by_purchase','recovered']].copy()
sched['effective_start'] = pd.to_datetime(sched['effective_start'], utc=True)
sched['effective_end']   = pd.to_datetime(sched['effective_end'],   utc=True)
# abandoned as bool for consistency
sched['abandoned'] = sched['abandoned'].astype(bool)

# --- 1) Row id for GA to merge back cleanly ---
ga_keyed = GA.reset_index().rename(columns={'index':'ga_row_id'})

# Keep only columns needed during the chunked join
ga_small    = ga_keyed[['ga_row_id','customer_id','event_ts_utc','event_name']].dropna(subset=['customer_id','event_ts_utc'])
sched_small = sched.dropna(subset=['customer_id','effective_start','effective_end'])

# --- 2) Batched merge by customer_id (memory friendly) ---
BATCH_SIZE = 2_000

all_ids  = ga_small['customer_id'].dropna().unique()
out_parts = []

for i in range(0, len(all_ids), BATCH_SIZE):
    ids = set(all_ids[i:i+BATCH_SIZE])

    # GA rows in this chunk
    ga_c = ga_small[ga_small['customer_id'].isin(ids)]
    if ga_c.empty:
        continue

    # Per-chunk GA time bounds by customer to prune schedule rows
    bounds = (
        ga_c.groupby('customer_id', as_index=False)
            .agg(ev_min=('event_ts_utc', 'min'),
                 ev_max=('event_ts_utc', 'max'))
    )

    # Only schedule rows whose window overlaps any GA timestamp range for that customer
    sc_c = (
        sched_small.merge(bounds, on='customer_id', how='inner')
                   .loc[lambda d: (d['effective_end']   >= d['ev_min']) &
                                  (d['effective_start'] <= d['ev_max'])]
                   [['customer_id','effective_start','effective_end',
                     'abandoned','purchase_segment','False_by_purchase','recovered']]
    )
    if sc_c.empty:
        continue

    # Candidate matches by customer_id
    m = ga_c.merge(sc_c, on='customer_id', how='left', copy=False)

    # Keep only rows where GA ts is inside the schedule window (inclusive)
    in_window = (
        (m['event_ts_utc'] >= m['effective_start']) &
        (m['event_ts_utc'] <= m['effective_end'])
    )
    m_win = m.loc[in_window, ['ga_row_id','event_name','effective_start',
                              'abandoned','purchase_segment','False_by_purchase','recovered']]
    if m_win.empty:
        continue

    # If a GA row matches multiple intervals, choose the one with the latest effective_start
    picked = (
        m_win.sort_values(['ga_row_id','effective_start'])
             .groupby('ga_row_id', as_index=False)
             .tail(1)[['ga_row_id','event_name','abandoned','purchase_segment','False_by_purchase','recovered']]
    )

    out_parts.append(picked)

# Combine flags from all chunks (may be empty if no matches found)
flag = (pd.concat(out_parts, ignore_index=True)
        if out_parts else
        pd.DataFrame(columns=['ga_row_id','event_name','abandoned','purchase_segment','False_by_purchase','recovered']))

# Keep only one row per GA row id (defensive)
flag = flag.sort_values('ga_row_id').drop_duplicates('ga_row_id', keep='last')

# --- 3) Bring fields back to GA (no new rows) ---
GA_enriched = (
    ga_keyed.merge(flag.drop(columns=['event_name']), on='ga_row_id', how='left')
            .drop(columns=['ga_row_id'])
)

# --- 4) Post-condition: a purchase event should never be marked abandoned=True ---
# (Effective_end already clamps via purchase_date in df, but enforce just in case.)
purchase_mask = GA_enriched['event_name'].astype(str).str.lower().eq('purchase')
GA_enriched.loc[purchase_mask, 'abandoned'] = False

# If you need the column named exactly "purchase_segement" (with the misspelling), create an alias:
if 'purchase_segment' in GA_enriched.columns:
    GA_enriched['purchase_segement'] = GA_enriched['purchase_segment']
    # (Optionally drop the correctly spelled version if you only want the alias)
    # GA_enriched = GA_enriched.drop(columns=['purchase_segment'])

# Result: GoogleAnalytics intact + appended columns:
# - abandoned (bool)
# - purchase_segment (and alias purchase_segement if requested)
# - False_by_purchase
# - recovered
# GA_enriched.head()


In [0]:
# import pandas as pd
# import numpy as np

# # --- 0) Copies and strict dtypes ---
# GA = GoogleAnalytics.copy()

# # strict datetime parsing: will raise if invalid formats
# GA['event_ts_utc'] = pd.to_datetime(GA['event_ts_utc'], utc=True)

# sched = dfRec[['customer_id',
#                'effective_start','effective_end',
#                'abandoned','purchase_segment','False_by_purchase','recovered']].copy()

# sched['effective_start'] = pd.to_datetime(sched['effective_start'], utc=True)
# sched['effective_end']   = pd.to_datetime(sched['effective_end'],   utc=True)

# sched['abandoned'] = sched['abandoned'].astype(bool)

# # align dtype strictly — will raise if incompatible
# GA['customer_id']    = GA['customer_id'].astype('int64')
# sched['customer_id'] = sched['customer_id'].astype('int64')

# # --- validations (fail fast) ---
# req_ga_cols = ['customer_id', 'event_ts_utc']
# req_sc_cols = ['customer_id', 'effective_start', 'effective_end']

# if GA[req_ga_cols].isna().any().any():
# #     bad = GA[GA[req_ga_cols].isna().any(axis=1)][req_ga_cols].ead()
#     raise ValueError(f"GA has nulls in required columns {req_ga_cols}. Sample:\n{bad}")

# if sched[req_sc_cols].isna().any().any():
#     bad = sched[sched[req_sc_cols].isna().any(axis=1)][req_sc_cols].head()
#     raise ValueError(f"sched has nulls in required columns {req_sc_cols}. Sample:\n{bad}")

# # --- 1) Row id for GA to merge back cleanly ---
# ga_keyed = GA.reset_index().rename(columns={'index': 'ga_row_id'})

# ga_small    = ga_keyed[['ga_row_id','customer_id','event_ts_utc','event_name']]
# sched_small = sched[['customer_id','effective_start','effective_end',
#                      'abandoned','purchase_segment','False_by_purchase','recovered']]

# # --- 2) Batched merge by customer_id ---
# BATCH_SIZE = 2_000

# all_ids   = ga_small['customer_id'].unique()
# out_parts = []

# for i in range(0, len(all_ids), BATCH_SIZE):
#     ids = set(all_ids[i:i+BATCH_SIZE])

#     # GA rows in this chunk
#     ga_c = ga_small[ga_small['customer_id'].isin(ids)]
#     if ga_c.empty:
#         continue

#     # Per-customer event bounds
#     bounds = (
#         ga_c.groupby('customer_id', as_index=False)['event_ts_utc']
#             .agg(ev_min='min', ev_max='max')
#     )

#     # Keep only overlapping schedule windows
#     sc_c = (
#         sched_small.merge(bounds, on='customer_id', how='inner')
#                    .loc[lambda d: (d['effective_end']   >= d['ev_min']) &
#                                   (d['effective_start'] <= d['ev_max'])]
#                    [['customer_id','effective_start','effective_end',
#                      'abandoned','purchase_segment','False_by_purchase','recovered']]
#     )
#     if sc_c.empty:
#         continue

#     # Candidate matches by customer_id
#     m = ga_c.merge(sc_c, on='customer_id', how='left', copy=False)

#     # Keep only rows inside the window
#     in_window = (
#         (m['event_ts_utc'] >= m['effective_start']) &
#         (m['event_ts_utc'] <= m['effective_end'])
#     )
#     m_win = m.loc[in_window, ['ga_row_id','event_name','effective_start',
#                               'abandoned','purchase_segment','False_by_purchase','recovered']]
#     if m_win.empty:
#         continue

#     # Choose one interval per GA row (latest effective_start)
#     picked = (
#         m_win.sort_values(['ga_row_id','effective_start'])
#              .groupby('ga_row_id', as_index=False)
#              .tail(1)[['ga_row_id','event_name','abandoned','purchase_segment','False_by_purchase','recovered']]
#     )

#     out_parts.append(picked)

# # Combine results
# flag = (pd.concat(out_parts, ignore_index=True)
#         if out_parts else
#         pd.DataFrame(columns=['ga_row_id','event_name','abandoned','purchase_segment','False_by_purchase','recovered']))

# # Deduplicate
# flag = flag.sort_values('ga_row_id').drop_duplicates('ga_row_id', keep='last')

# # --- 3) Bring fields back to GA ---
# GA_enriched = (
#     ga_keyed.merge(flag.drop(columns=['event_name']), on='ga_row_id', how='left')
#             .drop(columns=['ga_row_id'])
# )

# # --- optional alias for misspelled name ---
# if 'purchase_segment' in GA_enriched.columns:
#     GA_enriched['purchase_segement'] = GA_enriched['purchase_segment']

# # GA_enriched is your final enriched dataset, with no normalization, coercion, or overrides.


In [0]:
count2 = (GA_enriched['event_name'] == 'purchase').sum()
print(count2)
count1 = ((GA_enriched['False_by_purchase'] == 'no purchase') & (GA_enriched['event_name'] == 'purchase')).sum()
print(count1)
count3 = ((GA_enriched['False_by_purchase'] == 'purchased') & (GA_enriched['event_name'] == 'purchase')).sum()
print(count3)
count2 = (GoogleAnalytics['event_name'] == 'purchase').sum()
print(count2)

In [0]:
ga_with_abandoned = GA_enriched.sort_values(by=['customer_id', 'event_ts_utc'], ascending=[True, True])
ga_with_abandoned.head(n=5000)

In [0]:
ga_with_abandoned = ga_with_abandoned.drop(columns=['purchase_segment'])


In [0]:
ga_with_abandoned.to_csv('clean_GA.csv', index=False)


In [0]:
#GBCustomer = GoogleAnalytics.sort_values(by=['customer_id', 'event_ts_utc'], ascending=[True, True])
GBCustomer = ga_with_abandoned.sort_values(by=['False_by_purchase','event_name'], ascending=[True,True])

GBCustomer = GBCustomer[GBCustomer['event_name'] == 'purchase']

GBCustomer.head(n=10000)